In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from neo4j import GraphDatabase

In [ ]:
MODEL_PATH = "./flan_t5_text2cypher/"
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "password"

INSTRUCTION_PREFIX = (
    "Generate Cypher statement to query a graph database.\n"
    "Schema: {schema}\nQuestion: {question}\nCypher: "
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Model loaded on {device}")

In [ ]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Connected to Neo4j")

In [ ]:
def get_schema(driver):
    node_props = driver.execute_query(
        "CALL db.schema.nodeTypeProperties() YIELD nodeLabels, propertyName, propertyTypes "
        "RETURN nodeLabels, collect({property: propertyName, type: propertyTypes[0]}) AS properties"
    )[0]

    rel_props = driver.execute_query(
        "CALL db.schema.relTypeProperties() YIELD relType, propertyName, propertyTypes "
        "RETURN relType, collect({property: propertyName, type: propertyTypes[0]}) AS properties"
    )[0]

    rels = driver.execute_query(
        "CALL db.schema.visualization() YIELD nodes, relationships "
        "UNWIND relationships AS r "
        "RETURN labels(startNode(r))[0] AS start, type(r) AS type, labels(endNode(r))[0] AS end"
    )[0]

    schema_parts = []

    schema_parts.append("Node properties:")
    for record in node_props:
        labels = ":".join(record["nodeLabels"])
        props = ", ".join(f"{p['property']}: {p['type']}" for p in record["properties"] if p["property"])
        schema_parts.append(f"  ({labels} {{{props}}})")

    schema_parts.append("Relationship properties:")
    for record in rel_props:
        rel = record["relType"]
        props = ", ".join(f"{p['property']}: {p['type']}" for p in record["properties"] if p["property"])
        schema_parts.append(f"  {rel} {{{props}}}")

    schema_parts.append("Relationships:")
    for record in rels:
        schema_parts.append(f"  (:{record['start']})-[:{record['type']}]->(:{record['end']})")

    return "\n".join(schema_parts)

schema = get_schema(driver)
print(schema)

In [ ]:
def generate_cypher(question, schema, max_new_tokens=256):
    input_text = INSTRUCTION_PREFIX.format(schema=schema, question=question)
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=4096).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
question = "How many nodes are in the database?"
cypher = generate_cypher(question, schema)
print(f"Question: {question}")
print(f"Cypher:   {cypher}")

In [ ]:
# Run the generated query against Neo4j
records, summary, keys = driver.execute_query(cypher)
for record in records:
    print(record.data())